In [41]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [42]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("s1nghayush/data-dump")

print("Path to dataset files:", path)

Path to dataset files: /root/.cache/kagglehub/datasets/s1nghayush/data-dump/versions/1


In [43]:
df = pd.read_csv(path + '/ganjoo_ipl_mbm_bat_final.csv')
df.head()

,date,match_id,bat_team,bowl_team,inns,batter_id,batter,runs,balls,fours,sixes,sr,isOut,wicketType,bowler_id,bowler,venue
0,2015-04-08,829705,Mumbai Indians,Kolkata Knight Riders,1,277662,Corey Anderson,55,41,4,3,134.15,False,not out,NaN,NaN,Eden Gardens
1,2015-04-08,829705,Mumbai Indians,Kolkata Knight Riders,1,34102,Rohit Sharma,98,65,12,4,150.77,False,not out,NaN,NaN,Eden Gardens
2,2015-04-08,829705,Mumbai Indians,Kolkata Knight Riders,1,333904,Aditya Tare,7,7,1,0,100.00,True,caught,Shakib Al Hasan,Shakib Al Hasan,Eden Gardens
3,2015-04-08,829705,Mumbai Indians,Kolkata Knight Riders,1,5334,Aaron Finch,5,5,1,0,100.00,True,caught,Morne Morkel,Morne Morkel,Eden Gardens
4,2015-04-08,829705,Mumbai Indians,Kolkata Knight Riders,1,33141,Ambati Rayudu,0,2,0,0,0.00,True,caught,Morne Morkel,Morne Morkel,Eden Gardens


In [44]:
df['isOut'] = df['isOut'].replace('DNB', 'False')

In [45]:
df['isOut'] = df['isOut'].replace('True', True)
df['isOut'] = df['isOut'].replace('False', False)

<ipython-input-45-888660078a8b>:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['isOut'] = df['isOut'].replace('False', False)


In [46]:
# Define the conditions for each column
conditions = [
    (df['runs'] >= 25) & (df['runs'] < 50),
    (df['runs'] >= 50) & (df['runs'] < 75),
    (df['runs'] >= 75) & (df['runs'] < 100),
    (df['runs'] >= 100),
    (df['isOut'] & (df['runs'] == 0))  # 'isout' instead of 'isOut'
]

# Define the corresponding column names
column_names = ['25s', '50s', '75s', '100s', 'Duck']

# Initialize all new columns with 0
for col in column_names:
    df[col] = 0

# Assign 1 to the appropriate column based on conditions
for condition, col in zip(conditions, column_names):
    df.loc[condition, col] = 1


In [47]:
# Calculate batting fantasy points
df['batting_fp'] = (
    df['runs'] * 1 +  # Runs: +1 per run
    df['fours'] * 4 +  # Boundary Bonus: +4 per four
    df['sixes'] * 6 +  # Six Bonus: +6 per six
    df['25s'] * 4 +  # 25s: +4 pts
    df['50s'] * 8 +  # 50s: +8 pts
    df['75s'] * 12 +  # 75s: +12 pts
    df['100s'] * 16 +  # 100s: +16 pts
    df['Duck'] * -2  # Duck: -2 pts
)

# Add strike rate bonus/penalty
def strike_rate_points(row):
    if row['balls'] >= 10:  # Check if at least 10 balls were played
        if row['sr'] > 170:  # 'sr' instead of 'strikeRate'
            return 6
        elif 150.01 <= row['sr'] <= 170:
            return 4
        elif 130 <= row['sr'] <= 150:
            return 2
        elif 60 <= row['sr'] < 70:
            return -2
        elif 50 <= row['sr'] < 59.99:
            return -4
        elif row['sr'] < 50:
            return -6
    return 0  # Return 0 if fewer than 10 balls were played

# Apply strike rate points
df['strikeRate_points'] = df.apply(strike_rate_points, axis=1)
df['batting_fp'] += df['strikeRate_points']

# Drop the temporary strikeRate_points column if not needed
df.drop(columns=['strikeRate_points'], inplace=True)
df.head(10)


,date,match_id,bat_team,bowl_team,inns,batter_id,batter,runs,balls,fours,...,wicketType,bowler_id,bowler,venue,25s,50s,75s,100s,Duck,batting_fp
0,2015-04-08,829705,Mumbai Indians,Kolkata Knight Riders,1,277662,Corey Anderson,55,41,4,...,not out,NaN,NaN,Eden Gardens,0,1,0,0,0,99
1,2015-04-08,829705,Mumbai Indians,Kolkata Knight Riders,1,34102,Rohit Sharma,98,65,12,...,not out,NaN,NaN,Eden Gardens,0,0,1,0,0,186
2,2015-04-08,829705,Mumbai Indians,Kolkata Knight Riders,1,333904,Aditya Tare,7,7,1,...,caught,Shakib Al Hasan,Shakib Al Hasan,Eden Gardens,0,0,0,0,0,11
3,2015-04-08,829705,Mumbai Indians,Kolkata Knight Riders,1,5334,Aaron Finch,5,5,1,...,caught,Morne Morkel,Morne Morkel,Eden Gardens,0,0,0,0,0,9
4,2015-04-08,829705,Mumbai Indians,Kolkata Knight Riders,1,33141,Ambati Rayudu,0,2,0,...,caught,Morne Morkel,Morne Morkel,Eden Gardens,0,0,0,0,1,-2
5,2015-04-08,829705,Mumbai Indians,Kolkata Knight Riders,1,29264,Harbhajan Singh,0,0,0,...,DNB,NaN,NaN,Eden Gardens,0,0,0,0,0,0
6,2015-04-08,829705,Mumbai Indians,Kolkata Knight Riders,1,625383,Jasprit Bumrah,0,0,0,...,DNB,NaN,NaN,Eden Gardens,0,0,0,0,0,0
7,2015-04-08,829705,Mumbai Indians,Kolkata Knight Riders,1,230559,Kieron Pollard,0,0,0,...,DNB,NaN,NaN,Eden Gardens,0,0,0,0,0,0
8,2015-04-08,829705,Mumbai Indians,Kolkata Knight Riders,1,32130,Pragyan Ojha,0,0,0,...,DNB,NaN,NaN,Eden Gardens,0,0,0,0,0,0
9,2015-04-08,829705,Mumbai Indians,Kolkata Knight Riders,1,35731,Vinay Kumar,0,0,0,...,DNB,NaN,NaN,Eden Gardens,0,0,0,0,0,0


In [48]:
batting_fp = df.drop(columns=['inns', 'bat_team',  'isOut'])
batting_fp.head()

,date,match_id,bowl_team,batter_id,batter,runs,balls,fours,sixes,sr,wicketType,bowler_id,bowler,venue,25s,50s,75s,100s,Duck,batting_fp
0,2015-04-08,829705,Kolkata Knight Riders,277662,Corey Anderson,55,41,4,3,134.15,not out,NaN,NaN,Eden Gardens,0,1,0,0,0,99
1,2015-04-08,829705,Kolkata Knight Riders,34102,Rohit Sharma,98,65,12,4,150.77,not out,NaN,NaN,Eden Gardens,0,0,1,0,0,186
2,2015-04-08,829705,Kolkata Knight Riders,333904,Aditya Tare,7,7,1,0,100.00,caught,Shakib Al Hasan,Shakib Al Hasan,Eden Gardens,0,0,0,0,0,11
3,2015-04-08,829705,Kolkata Knight Riders,5334,Aaron Finch,5,5,1,0,100.00,caught,Morne Morkel,Morne Morkel,Eden Gardens,0,0,0,0,0,9
4,2015-04-08,829705,Kolkata Knight Riders,33141,Ambati Rayudu,0,2,0,0,0.00,caught,Morne Morkel,Morne Morkel,Eden Gardens,0,0,0,0,1,-2


In [49]:
batting_fp = df[['match_id', 'batter_id','batting_fp', 'date']].copy()
batting_fp

,match_id,batter_id,batting_fp,date
0,829705,277662,99,2015-04-08
1,829705,34102,186,2015-04-08
2,829705,333904,11,2015-04-08
3,829705,5334,9,2015-04-08
4,829705,33141,-2,2015-04-08
...,...,...,...,...
14263,1426312,1312645,0,2024-05-26
14264,1426312,311592,0,2024-05-26
14265,1426312,723105,0,2024-05-26
14266,1426312,1079470,0,2024-05-26


In [51]:
batting_fp.to_csv('ganjoo_ipl_bat_fp.csv', index=False)